In [17]:
import torch
import numpy as np
import pandas as pd
import os
import pathlib
import sys

from torchvision.transforms import v2
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import StepLR
from torchmetrics.detection.mean_ap import MeanAveragePrecision

import onnxruntime as ort

from pathlib import Path

import CarImageClass

from SSD_from_scratch import mySSD
from SSD_trainer import SSD_train, plot_losses, collate_detection, ConditionalIoUCrop, load_checkpoint, build_optimizer_and_scheduler

device = "cuda" if torch.cuda.is_available() else "cpu"

# set file path
# desktop or laptop
machine = 'desktop'

# Setup path to data folder
if machine == 'laptop':
    folder_path = Path(r"C:\self-driving-car\data")
else:
    folder_path = Path(r"C:\Udacity_car_data\data")

train_path = folder_path / "train"
test_path = folder_path / "test"

In [15]:
test_tfms = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Resize((300, 300), antialias=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406],
                 std=[0.229, 0.224, 0.225]),
])

test_data = CarImageClass.ImageClass(targ_dir=test_path, transform=test_tfms, file_pct=1)

In [16]:
len(test_data)

9937

In [28]:
pathlib.Path().resolve() / 'test'

WindowsPath('C:/Users/eblac/Documents/GitHub/self-driving-car/test')

In [29]:

# --- make repo root importable ---
REPO_ROOT = pathlib.Path().resolve() / 'PTQ_testing'
sys.path.insert(0, str(REPO_ROOT))

from calibration_data import build_calibration_loader_from_dataset, SSDCalibrationDataReader

# val_data already has transform_test=test_tfms per your split call
calib_loader = build_calibration_loader_from_dataset(
    test_data,
    num_samples=1000,   # or None to use all of val_data
    seed=724,
    batch_size=8,
    num_workers=0,
)

# sanity check
x = next(iter(calib_loader))
print(x.shape, x.dtype)  # expect [B,3,300,300], float32

calib_reader = SSDCalibrationDataReader(calib_loader, input_name="images")


torch.Size([8, 3, 300, 300]) torch.float32


In [2]:
ssd_model_noZO_BS = mySSD(class_to_idx_dict={'biker': 0, 'car': 1, 'pedestrian': 2, 'trafficLight': 3, 'truck': 4},
                          in_channels=3,
                          variances=(0.1, 0.2))

WEIGHTS_PATH = r"C:\Users\eblac\Documents\GitHub\self-driving-car\app_files\saved_models\noZoomOut_Bootstrap.pth"
state_dict = torch.load(WEIGHTS_PATH, map_location="cpu", weights_only=False)
res = ssd_model_noZO_BS.load_state_dict(state_dict, strict=False)
ssd_model_noZO_BS.to(device='cpu');

print("Missing keys:", res.missing_keys)
print("Unexpected keys:", res.unexpected_keys)

Missing keys: []
Unexpected keys: []


In [9]:
ssd_model_noZO_BS.eval()

dummy = torch.randn(1, 3, 300, 300)  # match your real input size
onnx_path = "ssd.onnx"

torch.onnx.export(
    ssd_model_noZO_BS,
    dummy,
    onnx_path,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["images"],
    output_names=["loc", "conf"],
    dynamic_axes={
        "images": {0: "batch"},
        "loc": {0: "batch"},
        "conf": {0: "batch"},
    },
)

In [3]:
sess = ort.InferenceSession(r"C:\Users\eblac\Documents\GitHub\self-driving-car\PTQ_testing\ssd.onnx", providers=["CPUExecutionProvider"])

def forward_ort(images_torch: torch.Tensor):
    # images_torch: [B,3,H,W], float32, already normalized exactly like training
    x = images_torch.detach().cpu().numpy().astype(np.float32)
    loc, conf = sess.run(["loc", "conf"], {"images": x})
    return loc, conf


In [4]:
def compare_outputs(torch_out, onnx_out, name: str, eps: float = 1e-12):
    """
    torch_out: torch.Tensor
    onnx_out: np.ndarray or torch.Tensor
    """
    if isinstance(onnx_out, torch.Tensor):
        onnx_np = onnx_out.detach().cpu().numpy()
    else:
        onnx_np = np.asarray(onnx_out)

    torch_np = torch_out.detach().cpu().numpy()

    assert torch_np.shape == onnx_np.shape, f"{name} shape mismatch: {torch_np.shape} vs {onnx_np.shape}"

    diff = torch_np - onnx_np
    abs_diff = np.abs(diff)

    max_abs = abs_diff.max()
    mean_abs = abs_diff.mean()
    rmse = np.sqrt((diff * diff).mean())

    denom = np.abs(torch_np) + eps
    rel = abs_diff / denom
    max_rel = rel.max()
    mean_rel = rel.mean()

    # also check for NaNs/Infs
    ok = np.isfinite(torch_np).all() and np.isfinite(onnx_np).all()

    print(f"[{name}] finite={ok}")
    print(f"[{name}] max_abs={max_abs:.3e}  mean_abs={mean_abs:.3e}  rmse={rmse:.3e}")
    print(f"[{name}] max_rel={max_rel:.3e}  mean_rel={mean_rel:.3e}")

def assert_close(torch_out, onnx_out, name: str, atol=1e-4, rtol=1e-3):
    if isinstance(onnx_out, np.ndarray):
        onnx_t = torch.from_numpy(onnx_out)
    else:
        onnx_t = onnx_out
    onnx_t = onnx_t.to(dtype=torch_out.dtype, device=torch_out.device)
    torch.testing.assert_close(torch_out, onnx_t, rtol=rtol, atol=atol)
    print(f"[{name}] PASSED torch.testing.assert_close(rtol={rtol}, atol={atol})")


In [7]:
torch.manual_seed(10)
np.random.seed(10)

ssd_model_noZO_BS.eval()
ssd_model_noZO_BS = ssd_model_noZO_BS.to("cpu").float()

x = torch.randn(1, 3, 300, 300, dtype=torch.float32)  # single shared input
x_np = x.numpy()  # same bytes, same values

with torch.inference_mode():
    loc_t, conf_t = ssd_model_noZO_BS(x)  # torch tensors

loc_o, conf_o = sess.run(["loc", "conf"], {"images": x_np})

compare_outputs(loc_t,  loc_o,  "loc")
compare_outputs(conf_t, conf_o, "conf")

assert_close(loc_t.cpu(),  loc_o,  "loc",  atol=1e-4, rtol=1e-3)
assert_close(conf_t.cpu(), conf_o, "conf", atol=1e-4, rtol=1e-3)


[loc] finite=True
[loc] max_abs=1.192e-06  mean_abs=9.214e-08  rmse=1.329e-07
[loc] max_rel=1.942e+00  mean_rel=5.884e-05
[conf] finite=True
[conf] max_abs=2.861e-06  mean_abs=1.532e-07  rmse=2.567e-07
[conf] max_rel=1.178e-04  mean_rel=1.027e-07
[loc] PASSED torch.testing.assert_close(rtol=0.001, atol=0.0001)
[conf] PASSED torch.testing.assert_close(rtol=0.001, atol=0.0001)
